In [4]:
import os

from conda.exports import root_dir
from dask.config import config
from joblib.externals.loky.backend.spawn import prepare

In [5]:
%pwd

'C:\\Users\\Lenovo\\image classification\\image-classification-\\research'

In [6]:
os.chdir("../")

In [7]:
%pwd

'C:\\Users\\Lenovo\\image classification\\image-classification-'

In [8]:
from dataclasses import dataclass
from pathlib import  Path
@dataclass (frozen=True)
class PrepareBaseModelConfig :
    root_dir : Path
    base_model_path  : Path
    updated_base_model_path : Path
    params_image_size : list
    params_learning_rate : float
    params_include_top : bool
    params_weight : str
    params_classes : int

In [9]:
from cnnclassifier.constants import *
from cnnclassifier.utils.common import read_yaml , create_directories

In [10]:
class ConfigurationManager :
    def __init__(
    self ,
    config_filepath = Config_File_path  ,
    params_filepath = Paramas_File_path) :

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_prepare_base_model_config(self) -> PrepareBaseModelConfig :
        config = self.config.prepare_base_model
        create_directories ([config.root_dir])

        prepare_base_model_config = PrepareBaseModelConfig(
            root_dir = Path(config.root_dir),
            base_model_path = Path(config.base_model_path),
            updated_base_model_path = Path(config.updated_base_model_path),
            params_image_size = self.params.IMAGE_SIZE,
            params_learning_rate = self.params.LEARNING_RATE,
            params_classes = self.params.CLASSES,
            params_weight = self.params.WEIGHT,
            params_include_top = self.params.INCLUDE_TOP,


        )
        return prepare_base_model_config



In [11]:
import  os
import  urllib.request as  request
from zipfile import  ZipFile
import  tensorflow as tf

In [12]:
class PrepareBaseModel :
    def __init__(self , config : PrepareBaseModelConfig) :
        self.config = config


    def get_base_model (self) :
        self.model = tf.keras.applications.vgg16.VGG16(

            input_shape = self.config.params_image_size,
            weights= self.config.params_weight ,
            include_top=self.config.params_include_top ,
            # output_shape = self.config.params_classes ,
        )

        self.save_model(path = self.config.base_model_path , model = self.model)

    @staticmethod
    def prepare_ful_model (model , classes , freeze_all , freeze_till , learning_rate) :
        if freeze_all :
            for layer in model.layers :
                layer.trainable = False
        elif (freeze_till is not None) and (freeze_till == 0) :
            for layer in model.layers :
                layer.trainable = False
        flatten_in  = tf.keras.layers.Flatten()(model.output)
        prediction = tf.keras.layers.Dense(units=classes , activation="softmax")(flatten_in)

        full_model = tf.keras.Model(
            inputs = model.input,
            outputs = prediction,
        )

        full_model.compile(
            optimizer =  tf.keras.optimizers.Adam(learning_rate=learning_rate) ,
            loss = tf.keras.losses.CategoricalCrossentropy(),
            metrics = ['accuracy']

        )

        full_model.summary()
        return full_model

    def update_base_model (self) :
        self.full_model = self.prepare_ful_model(
            model = self.model,
            classes = self.config.params_classes ,
            freeze_all= True ,
            learning_rate= self.config.params_learning_rate ,
            freeze_till= None ,

        )
        self.save_model(path = self.config.updated_base_model_path , model = self.full_model)

    @staticmethod
    def save_model(path = Path , model = tf.keras.Model ) :
        model.save(path)

In [13]:
try :
    config = ConfigurationManager()
    prepare_base_model_config = config.get_prepare_base_model_config()
    prepare_base_model = PrepareBaseModel(config = prepare_base_model_config)
    prepare_base_model.get_base_model()
    prepare_base_model.update_base_model()
except Exception as e :
    raise e


[2026-07-22 18:12:15,437 : INFO : common : yaml file : config\config.yaml loaded successfully]
[2026-07-22 18:12:15,450 : INFO : common : yaml file : params.yaml loaded successfully]
[2026-07-22 18:12:15,454 : INFO : common : created directory in  : artifacts]
[2026-07-22 18:12:15,456 : INFO : common : created directory in  : artifacts/prepare_base_model]
[2026-07-22 18:12:16,215 : WARNING : saving_api : You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. ]
[2026-07-22 18:12:16,364 : WARNING : config : TensorFlow GPU support is not available on native Windows for TensorFlow >= 2.11. Even if CUDA/cuDNN are installed, GPU will not be used. Please use WSL2 or the TensorFlow-DirectML plugin.]


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 224, 224, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 224, 224, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 112, 112, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 112, 112, 128)  │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 56, 56, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 28, 28, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 28, 28, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 14, 14, 512)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 7, 7, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 2)              │        50,178 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,764,866 (56.32 MB)

 Trainable params: 50,178 (196.01 KB)

 Non-trainable params: 14,714,688 (56.13 MB)

[2026-07-22 18:12:16,408 : WARNING : saving_api : You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. ]
